In [13]:
import psutil
import json
import os
import math
import gc
import random
import pathlib
import collections 
from typing import NamedTuple

import joblib
import tqdm.auto
import transformers
import torch
import plotly.express
import sklearn.decomposition
import matplotlib.pyplot as plt
import polars as pl
import numpy as np
import safetensors
import safetensors.torch
import plotly.express
import plotly.graph_objects

from lovely_tensors import lovely
from torch import Tensor

from IPython.display import clear_output

In [14]:
class ModelInfo(NamedTuple):
    tag: str
    name: str
    arange: torch.Tensor
    tokenizer: transformers.PreTrainedTokenizerFast | None
    ids: torch.Tensor | None
    embs: torch.Tensor
    display_name: str


def load_model(tag: str, model_name: str, tokens: list[str], shuffle: bool, cache_file: pathlib.Path, display_name: str) -> ModelInfo:
    tokenizer: transformers.PreTrainedTokenizerFast = transformers.AutoTokenizer.from_pretrained(model_name)
    ids = [tokenizer.convert_tokens_to_ids(str(i)) for i in tokens]
    
    if cache_file.exists():
        with safetensors.safe_open(cache_file, framework="pt") as f:
            embs = f.get_tensor("embs")
    else:
        model: transformers.PreTrainedModel = transformers.AutoModel.from_pretrained(model_name).eval()
        embs = model.get_input_embeddings().weight[ids].detach().clone()
        del model
        gc.collect()
        safetensors.torch.save_file({"embs": embs}, cache_file)

    if shuffle:
        embs = embs[torch.randperm(len(embs), generator=torch.Generator().manual_seed(0))]
    
    return ModelInfo(tag=tag, name=model_name, arange=tokens, tokenizer=tokenizer, ids=None, embs=embs, display_name=display_name)


def load_standard_gaussian(tag: str, model_name: str, arange: list[str], display_name: str) -> ModelInfo:
    model_dim = transformers.AutoConfig.from_pretrained(model_name).hidden_size
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    rng_gen = torch.Generator().manual_seed(0)
    shape = (len(arange), model_dim)
    embs = torch.normal(mean=torch.zeros(shape), std=torch.ones(shape), generator=rng_gen)
    return ModelInfo(tag=tag, name=model_name, arange=arange, tokenizer=tokenizer, ids=None, embs=embs, display_name=display_name)


In [ ]:
names_ranges = [
    ("olmo1b", "allenai/OLMo-2-0425-1B", 1000, "OLMo 2 1B"),
    ("olmo7b", "allenai/OLMo-2-1124-7B", 1000, "OLMo 2 7B"),
    ("olmo13b", "allenai/OLMo-2-1124-13B", 1000, "OLMo 2 13B"),
    #("olmo32b", "allenai/OLMo-2-0325-32B", 1000, "OLMo 2 32B"),

    ("llama1b", "meta-llama/Llama-3.2-1B", 1000, "Llama 3 1B"),
    ("llama3b", "meta-llama/Llama-3.2-3B", 1000, "Llama 3 3B"),
    ("llama8b", "meta-llama/Llama-3.1-8B", 1000, "Llama 3 8B"),
    #("llama70b", "meta-llama/Llama-3.1-70B", 1000, "Llama 3 70B"),
    
    #("phi", "microsoft/phi-4", 1000, "Phi 4 15B"),
    #("phi-mini", "microsoft/Phi-4-mini-instruct", 1000, "Phi 4 4B"),
]

cache_dir = pathlib.Path("model_embs_cache")
cache_dir.mkdir(exist_ok=True)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
nums = ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten"]

models_days: dict[str, ModelInfo] = {}
models_months: dict[str, ModelInfo] = {}
models_nums: dict[str, ModelInfo] = {}

for nick, ckpt_name, arange, display_name in tqdm.tqdm(names_ranges):
    print(nick)

    cache_filename_days = cache_dir / (ckpt_name.replace("/", "__") + "_days" + ".safetensors")
    models_days[nick] = load_model("pretrained", ckpt_name, days, shuffle=False, cache_file=cache_filename_days, display_name=display_name)
    models_days["shuffled_" + nick] = load_model("shuffled", ckpt_name, days, shuffle=True, cache_file=cache_filename_days, display_name="Shuffled " + display_name)
    models_days["sgn_" + nick] = load_standard_gaussian("sgn", ckpt_name, days, display_name="Random " + display_name)

    cache_filename_months = cache_dir / (ckpt_name.replace("/", "__") + "_months" + ".safetensors")
    models_months[nick] = load_model("pretrained", ckpt_name, months, shuffle=False, cache_file=cache_filename_months, display_name=display_name)
    models_months["shuffled_" + nick] = load_model("shuffled", ckpt_name, months, shuffle=True, cache_file=cache_filename_months, display_name="Shuffled " + display_name)
    models_months["sgn_" + nick] = load_standard_gaussian("sgn", ckpt_name, months, display_name="Random " + display_name)

    cache_filename_nums = cache_dir / (ckpt_name.replace("/", "__") + "_nums_str" + ".safetensors")
    models_nums[nick] = load_model("pretrained", ckpt_name, nums, shuffle=False, cache_file=cache_filename_nums, display_name=display_name)
    models_nums["shuffled_" + nick] = load_model("shuffled", ckpt_name, nums, shuffle=True, cache_file=cache_filename_nums, display_name="Shuffled " + display_name)
    models_nums["sgn_" + nick] = load_standard_gaussian("sgn", ckpt_name, nums, display_name="Random " + display_name)

  0%|          | 0/6 [00:00<?, ?it/s]

olmo1b


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 17%|█▋        | 1/6 [00:10<00:54, 11.00s/it]

olmo7b


...

 67%|██████▋   | 4/6 [01:07<00:32, 16.41s/it]

llama3b


 83%|████████▎ | 5/6 [01:15<00:13, 13.12s/it]

llama8b


100%|██████████| 6/6 [01:24<00:00, 14.03s/it]


In [25]:
def pca(embs: Tensor, low_dim: int) -> tuple[Tensor, Tensor]:
    pca = sklearn.decomposition.PCA(n_components=low_dim)
    reduced_embs = pca.fit_transform(embs.detach().numpy())
    return torch.tensor(reduced_embs), torch.tensor(pca.explained_variance_ratio_)

def fourier(embs: Tensor) -> Tensor:
    return torch.fft.fft(embs, dim=0)

def vis_emb_plotly(embs: Tensor, title: str) -> plotly.graph_objects.Figure:
    fig = plotly.express.imshow(
        embs.cpu().T.detach(),
        color_continuous_scale="blues",
        aspect='auto',
    )
    return fig.update_layout(title=title).update_xaxes(title="Token Value").update_yaxes(title="Feature")

In [26]:
import matplotlib
import PIL.Image

def vis_emb(embs: Tensor, colorful: bool) -> PIL.Image.Image:
    x = embs.cpu().detach()
    x_normalized = (x - x.min()) / (x.max() - x.min())
    theme = matplotlib.colormaps["Blues"]
    if colorful:
        x_normalized = theme(x_normalized)
    else:
        x_normalized = x_normalized.numpy()
    vis = PIL.Image.fromarray((x_normalized * 255).astype(np.uint8))
    return vis

#vis_emb(models_days["llama1b"].embs.T, colorful=False)

In [27]:
# make 2-component pca from embeddings and draw a scatterplot
model = models_months["llama8b"]
pca_embs, var_ratio = pca(model.embs, low_dim=2)
pca_embs /= pca_embs.norm(dim=1, keepdim=True)
# make aspect ratio square
fig = plotly.express.scatter(x=pca_embs[:, 0].cpu(), y=pca_embs[:, 1].cpu(), text=model.arange).update_xaxes(title="PC 1").update_yaxes(title="PC 2", scaleanchor="x")
fig

In [28]:
# make 2-component pca from embeddings and draw a scatterplot
model = models_days["llama1b"]
pca_embs, var_ratio = pca(model.embs, low_dim=2)
pca_embs /= pca_embs.norm(dim=1, keepdim=True)
fig = plotly.express.scatter(x=pca_embs[:, 0].cpu(), y=pca_embs[:, 1].cpu(), text=model.arange).update_yaxes(scaleanchor="x")
fig

In [46]:
# make 2-component pca from embeddings and draw a scatterplot
model = models_nums["olmo7b"]
pca_embs, var_ratio = pca(model.embs, low_dim=2)
pca_embs /= pca_embs.norm(dim=1, keepdim=True)
fig = plotly.express.scatter(x=pca_embs[:, 0].cpu(), y=pca_embs[:, 1].cpu(), text=model.arange).update_yaxes(scaleanchor="x")
fig

In [ ]:
def sinusoidal_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int,
    max_value: int,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    """
    Encodes a tensor of numbers into a sinusoidal representation, inspired by how absolute positional
    encoding works in transformers.

    The encoding is an evaluation of a sine and cosine function at different frequencies, where the
    frequency is determined by the embedding dimension and the allowed range of the input values.

    >>> sinusoidal_encode(
    ...     torch.tensor([-5, 2, 1, 0]),
    ...     embedding_dim=6,
    ...     min_value=-5,
    ...     max_value=5,
    ... )
    tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
            [ 0.6570,  0.7539, -0.1073, -0.9942,  0.9980,  0.0627],
            [-0.2794,  0.9602,  0.3491, -0.9371,  0.9616,  0.2746],
            [-0.9589,  0.2837,  0.7317, -0.6816,  0.8806,  0.4738]])
    """

    if embedding_dim % 2 != 0 and not use_l2_norm:
        raise ValueError("Embedding dimension must be even")

    if use_l2_norm:
        if embedding_dim % 2 == 0:
            reserved_dim = 2
        else:
            reserved_dim = 1
        embedding_dim -= reserved_dim
    else:
        reserved_dim = 0  # will not be used

    domain = max_value - min_value
    y_shape = x.shape + (embedding_dim,)
    y = torch.zeros(y_shape, device=x.device)
    even_indices = torch.arange(0, embedding_dim, 2)
    log_term = torch.log(torch.tensor(domain)) / embedding_dim
    div_term = torch.exp(even_indices * -log_term)
    x = x - min_value
    values = x.unsqueeze(-1).float() * div_term
    y[..., 0::2] = torch.sin(values)
    y[..., 1::2] = torch.cos(values)

    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserved_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)

    if norm_const is not None:
        y *= norm_const

    return y



class ClassifierProbe(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, basis: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.basis_to_latent = torch.nn.Linear(basis.shape[-1], hidden_dim, bias=True)
        self.basis: torch.nn.Buffer
        self.heldout_mask: torch.nn.Buffer
        self.register_buffer("basis", basis)
        self.register_buffer("heldout_mask", heldout_mask)
        
    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        latent_choices = self.basis_to_latent(self.basis)
        logits = latent_x @ latent_choices.T
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = float("-inf")
        return logits
    
class SinProbeNew(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, choices: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.freqs = torch.nn.Parameter(torch.linspace(1/(choices.max() - choices.min()), 0.5, steps=hidden_dim))
        self.phases = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.amplitudes = torch.nn.Parameter(torch.ones(hidden_dim) * 0.0001)
        # self.accels = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.hidden_dim = hidden_dim
        self.heldout_mask: torch.nn.Buffer
        self.choices: torch.nn.Buffer
        self.register_buffer("heldout_mask", heldout_mask)
        self.register_buffer("choices", choices)

    def get_waves(self) -> Tensor:
        waves = torch.sin(
            self.phases.unsqueeze(1)
            + (2 * torch.pi * self.freqs.unsqueeze(1) * self.choices.unsqueeze(0))
            # + (2 * torch.pi * self.accels.unsqueeze(1) * torch.log(self.choices.unsqueeze(0) + 1e-4))
        )
        # sort by frequency
        # waves = waves[torch.argsort(self.freqs.abs()), :]
        # assert waves.shape == (self.hidden_dim, len(self.choices))
        return waves * self.amplitudes.unsqueeze(1)

    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        waves = self.get_waves()
        logits = latent_x @ waves

        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = -torch.inf
        return logits

In [31]:
N_STEPS = 300
LR = 1e-3
L1_REG = 0.001
L2_REG = 0.01
HIDDEN_DIM = 512

In [ ]:
accs_history_sin = {}

for items_name, items, models in [("nums", nums, models_nums), ("days", days, models_days), ("months", months, models_months)]:
    accs_history_sin[items_name] = {}
    for nickname, model in models.items():
        accs_history_sin[items_name][nickname] = []
        if "shuffled" in nickname or "sgn" in nickname:
            continue
        
        embs = model.embs

        basis = sinusoidal_encode(
            torch.arange(len(items)),
            embedding_dim=embs.shape[1],
            min_value=0,
            max_value=len(items) - 1,
        )

        log_every_n = 1

        torch.manual_seed(0)
        heldout_masks = torch.eye(len(items), dtype=torch.bool)
        probes = [
            ClassifierProbe(emb_dim=embs.shape[1], hidden_dim=HIDDEN_DIM, basis=basis, heldout_mask=heldout_masks[i])
            for i in range(len(items))
        ]
        optims = [
            torch.optim.Adam(probe.parameters(), lr=LR) for probe in probes
        ]
        accs_history_sin[items_name][nickname] = []
        for step in range(N_STEPS):
            for heldout_idx in range(len(items)):
                heldout_mask = heldout_masks[heldout_idx]
                probe = probes[heldout_idx]
                optim = optims[heldout_idx]
                optim.zero_grad()
                y = torch.arange(len(items))[~heldout_mask]
                logits = probe(embs[~heldout_mask], holdout_eval_tokens=True)
                loss = torch.nn.functional.cross_entropy(logits, y)
                loss += L2_REG * probe.emb_to_latent.weight.norm() + L1_REG * probe.basis_to_latent.weight.abs().sum()
                loss.backward()
                optim.step()
            if step % log_every_n == 0:
                accs = []
                probs = []
                for heldout_idx in range(len(items)):
                    probe = probes[heldout_idx]
                    logits = probe(embs[heldout_idx], holdout_eval_tokens=False)
                    prediction = logits.argmax(dim=-1)
                    prob = torch.nn.functional.softmax(logits, dim=-1)[heldout_idx]
                    accs.append((prediction == heldout_idx).item())
                    probs.append(prob.item())
                accs_history_sin[items_name][nickname].append(accs)
                clear_output(wait=True)
                plotly.express.line(
                    x=list(range(len(accs_history_sin[items_name][nickname]))),
                    y=torch.tensor(accs_history_sin[items_name][nickname]).float().mean(dim=1),
                    range_x=[0, len(accs_history_sin[items_name][nickname])],
                    range_y=[0, 1],
                    title=f"Probe Accuracy for {items_name} {model.display_name}",
                ).show()

clear_output(wait=True)

In [33]:
import pandas as pd
for items_name in accs_history_sin.keys():
    accs = pd.DataFrame.from_dict({k: accs_history_sin[items_name][k] for k in accs_history_sin[items_name].keys() if not "shuffled" in k and not "sgn" in k}, orient="index").T.map(sum)
    plotly.express.line(
        accs,
        title=f"Probe Accuracy for {items_name}"
    ).show()
    print(f"Max accuracy for {items_name}")
    display(accs.max())

Max accuracy for nums


olmo1b     3
olmo7b     4
olmo13b    4
llama1b    3
llama3b    4
llama8b    3
dtype: int64

Max accuracy for days


olmo1b     4
olmo7b     6
olmo13b    5
llama1b    5
llama3b    7
llama8b    3
dtype: int64

Max accuracy for months


olmo1b      6
olmo7b      6
olmo13b     5
llama1b     9
llama3b    10
llama8b     6
dtype: int64

In [34]:
accs_history_sin_param = {}

for items_name, items, models in [("nums", nums, models_nums), ("days", days, models_days), ("months", months, models_months)]:
    accs_history_sin_param[items_name] = {}

    for nickname, model in models.items():
        accs_history_sin_param[items_name][nickname] = []
        if "shuffled" in nickname or "sgn" in nickname:
            continue
        
        embs = model.embs
        log_every_n = 1

        torch.manual_seed(0)
        heldout_masks = torch.eye(len(items), dtype=torch.bool)
        probes = [
            SinProbeNew(emb_dim=embs.shape[1], hidden_dim=HIDDEN_DIM, choices=torch.arange(len(items)), heldout_mask=heldout_masks[i])
            for i in range(len(items))
        ]
        optims = [
            torch.optim.Adam(probe.parameters(), lr=LR) for probe in probes
        ]
        accs_history_sin_param[items_name][nickname] = []
        for step in range(N_STEPS):
            for heldout_idx in range(len(items)):
                heldout_mask = heldout_masks[heldout_idx]
                probe = probes[heldout_idx]
                optim = optims[heldout_idx]
                optim.zero_grad()
                y = torch.arange(len(items))[~heldout_mask]
                logits = probe(embs[~heldout_mask], holdout_eval_tokens=True)
                loss = torch.nn.functional.cross_entropy(logits, y)
                loss += L2_REG * probe.emb_to_latent.weight.norm() + L1_REG * probe.amplitudes.abs().sum()
                loss.backward()
                optim.step()
            if step % log_every_n == 0:
                accs = []
                probs = []
                for heldout_idx in range(len(items)):
                    probe = probes[heldout_idx]
                    logits = probe(embs[heldout_idx], holdout_eval_tokens=False)
                    prediction = logits.argmax(dim=-1)
                    prob = torch.nn.functional.softmax(logits, dim=-1)[heldout_idx]
                    accs.append((prediction == heldout_idx).item())
                    probs.append(prob.item())
                accs_history_sin_param[items_name][nickname].append(accs)
                clear_output(wait=True)
                plotly.express.line(
                    x=list(range(len(accs_history_sin_param[items_name][nickname]))),
                    y=torch.tensor(accs_history_sin_param[items_name][nickname]).float().mean(dim=1),
                    range_x=[0, len(accs_history_sin_param[items_name][nickname])],
                    range_y=[0, 1],
                    title=f"Probe Accuracy for {items_name} {model.display_name}",
                ).show()

clear_output(wait=True)

In [35]:
import pandas as pd
for items_name in accs_history_sin_param.keys():
    accs = pd.DataFrame.from_dict({k: accs_history_sin_param[items_name][k] for k in accs_history_sin_param[items_name].keys() if not "shuffled" in k and not "sgn" in k}, orient="index").T.map(sum)
    plotly.express.line(
        accs,
        title=f"Probe Accuracy for {items_name}"
    ).show()
    print(f"Max accuracy for {items_name}")
    display(accs.max())

Max accuracy for nums


olmo1b     6
olmo7b     8
olmo13b    8
llama1b    5
llama3b    6
llama8b    9
dtype: int64

Max accuracy for days


olmo1b     6
olmo7b     6
olmo13b    6
llama1b    6
llama3b    6
llama8b    6
dtype: int64

Max accuracy for months


olmo1b     11
olmo7b     11
olmo13b    11
llama1b    11
llama3b    11
llama8b    11
dtype: int64

In [36]:
for nick, accs in accs_history_sin_param["days"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo7b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo13b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama1b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama3b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama8b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)


In [37]:
for nick, accs in accs_history_sin_param["months"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo7b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo13b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama1b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama3b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama8b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)


In [38]:
for nick, accs in accs_history_sin_param["nums"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([0, 1, 0, 0, 0, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo7b tensor([0, 1, 1, 1, 0, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo13b tensor([0, 1, 1, 0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama1b tensor([0, 1, 0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.int32)
llama3b tensor([0, 1, 0, 0, 1, 0, 1, 1, 1, 1], dtype=torch.int32)
llama8b tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)


In [39]:
for nick, accs in accs_history_sin["days"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo7b tensor([1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo13b tensor([1, 1, 1, 1, 1, 0, 1], dtype=torch.int32)
llama1b tensor([0, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama3b tensor([1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama8b tensor([1, 1, 1, 1, 1, 1, 0], dtype=torch.int32)


In [40]:
for nick, accs in accs_history_sin["months"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
olmo7b tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1], dtype=torch.int32)
olmo13b tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], dtype=torch.int32)
llama1b tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], dtype=torch.int32)
llama3b tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)
llama8b tensor([1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1], dtype=torch.int32)


In [41]:
for nick, accs in accs_history_sin["nums"].items():
    if "shuffled" in nick or "sgn" in nick:
        continue
    print(nick, torch.tensor(accs).max(dim=0).values.int())

olmo1b tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0], dtype=torch.int32)
olmo7b tensor([1, 0, 1, 1, 1, 1, 1, 1, 0, 1], dtype=torch.int32)
olmo13b tensor([1, 0, 1, 1, 1, 1, 1, 1, 0, 0], dtype=torch.int32)
llama1b tensor([0, 1, 1, 1, 1, 1, 1, 1, 0, 0], dtype=torch.int32)
llama3b tensor([0, 1, 1, 1, 1, 1, 1, 0, 0, 0], dtype=torch.int32)
llama8b tensor([1, 1, 1, 1, 1, 1, 0, 0, 1, 0], dtype=torch.int32)
